In [1]:
import pandas as pd
from pandas.api.types import is_numeric_dtype, is_string_dtype
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score,mean_squared_error, r2_score, mean_absolute_error, f1_score, roc_auc_score
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit, train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [2]:
movies = pd.read_csv("../data/processed/movies.csv")
casting = pd.read_csv("../data/processed/casting.csv")

## Person

In [3]:
person_raw = movies.merge(casting, left_on="movie_id", right_on = "Movie ID", how="inner")
person_raw["profitable_movie_id"] = person_raw["Movie ID"].where(person_raw["Profit (USD)"] > 0)
person_raw["fin_available_movie_id"] = person_raw["Movie ID"].where(person_raw["Revenue (USD)"].notna() & person_raw["Budget (USD)"].notna())
person_raw = person_raw.sort_values(by="Release Date", ascending=True)
person_raw["blockbuster_movie_id"] = person_raw["Movie ID"].where(person_raw["Blockbuster"] == True)

person = person_raw.groupby("pid").agg(
    pid = ("pid", "first"),
    Name = ("Name", "first"),
    Gender = ("Gender", "first"), 
    n_movies = ("Movie ID", "nunique"),
    wavg_rating = ("Weighted Rating", "mean"),
    avg_mpop = ("Popularity_x", "mean"), 
    top3_mpop = ("Popularity_x", lambda x: x.nlargest(3).mean()),
    recent_mpop = ("Popularity_x", lambda x: x.iloc[-1] if not x.empty else np.nan),
    avg_pop = ("Popularity_y", "mean"),
    n_fin = ("fin_available_movie_id", "nunique"),
    n_blockbuster = ("blockbuster_movie_id", "nunique"),
    n_profit = ("profitable_movie_id", "nunique"),
    first_yr = ("Release Date", lambda x: pd.to_datetime(x).dt.year.min()),
    last_yr = ("Release Date", lambda x: pd.to_datetime(x).dt.year.max()),
    main_job = ("Job", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

## Person_Movie_History

In [4]:
# Making Past History
person_raw = movies.merge(casting, left_on="movie_id", right_on = "Movie ID", how="inner")
person_raw["profitable_movie_id"] = person_raw["Movie ID"].where(person_raw["Profit (USD)"] > 0)
person_raw["fin_available_movie_id"] = person_raw["Movie ID"].where(person_raw["Revenue (USD)"].notna() & person_raw["Budget (USD)"].notna())
person_raw = person_raw.sort_values(by="Release Date", ascending=True)
person_raw["blockbuster_movie_id"] = person_raw["Movie ID"].where(person_raw["Blockbuster"] == True)

df = person_raw.copy()

# Make sure release date is datetime
df["Release Date"] = pd.to_datetime(df["Release Date"], errors="coerce")

# Sort first, because "recent" depends on time order
df = df.sort_values(["pid", "Release Date", "Movie ID"]).copy()

In [5]:
def make_past_person_features(g):
    """
    For one person, calculate historical features for each movie row.
    Only movies released before the current row's release date are used.
    """
    g = g.sort_values(["Release Date", "Movie ID"]).copy()

    output_rows = []

    for current_date, current_rows in g.groupby("Release Date", sort=True):
        past = g[g["Release Date"] < current_date]

        if past.empty:
            features = {
                "prev_n_movies": 0,
                "prev_wavg_rating": np.nan,
                "prev_avg_mpop": np.nan,
                "prev_top3_mpop": np.nan,
                "prev_recent_mpop": np.nan,
                "prev_avg_pop": np.nan,
                "prev_n_fin": 0,
                "prev_n_blockbuster": 0,
                "prev_n_profit": 0,
                "prev_first_yr": np.nan,
                "prev_last_yr": np.nan,
                "prev_main_job": np.nan,
            }
        else:
            job_mode = past["Job"].mode()

            features = {
                "prev_n_movies": past["Movie ID"].nunique(),

                "prev_wavg_rating": past["Weighted Rating"].mean(),

                "prev_avg_mpop": past["Popularity_x"].mean(),

                "prev_top3_mpop": past["Popularity_x"].nlargest(3).mean(),

                # Most recent previous movie popularity
                "prev_recent_mpop": (
                    past.sort_values(["Release Date", "Movie ID"])["Popularity_x"].iloc[-1]
                ),

                # Be careful with this if Popularity_y is current TMDB person popularity
                "prev_avg_pop": past["Popularity_y"].mean(),

                "prev_n_fin": past["fin_available_movie_id"].nunique(),

                "prev_n_blockbuster": past["blockbuster_movie_id"].nunique(),

                "prev_n_profit": past["profitable_movie_id"].nunique(),

                "prev_first_yr": past["Release Date"].dt.year.min(),

                "prev_last_yr": past["Release Date"].dt.year.max(),

                "prev_main_job": job_mode.iloc[0] if not job_mode.empty else np.nan,
            }

        current_rows = current_rows.copy()

        for col, value in features.items():
            current_rows[col] = value

        output_rows.append(current_rows)

    return pd.concat(output_rows, ignore_index=True)

In [19]:
person_movie_history = (
    df.groupby("pid", group_keys=False)
      .apply(make_past_person_features)
      .reset_index(drop=True)
)

person_movie_history["prev_career_span"] = person_movie_history["prev_last_yr"] - person_movie_history["prev_first_yr"] + 1

## Movie Extended (With Person History)

In [20]:
def make_movies_cast_crew_statistics(g):
    """
    For each movie, calculate the Cast and Crew statistics based on the person's past movies.
    Only movies released before the current row's release date are used.
    """

    output_rows = []

    cast = g[g["Job"] == "Acting"]
    cast["prev_experienced"] = cast["prev_n_movies"] > 3

    crew = g[g["Job"] != "Acting"]
    crew["prev_experienced"] = crew["prev_n_movies"] > 5

    director = g[g["Job"] == "Director"]
    director["prev_experienced"] = director["prev_n_movies"] > 3

    features = {
        "Movie ID": g.name,

        "has_cast": 0,
        "top_cast_pop": 0,
        "top_cast_career_span": 0,

        "top5_cast_pop": 0,
        "top5_cast_wavg_mrating": 0,
        "top5_cast_avg_mpop": 0,
        "top5_cast_nblockbuster": 0,
        "top5_cast_nprofit": 0,
        "top5_cast_career_span": 0,
        "n_experienced_cast": 0,

        "has_crew": 0,
        "top5_crew_wavg_mrating": 0,
        "top5_crew_avg_mpop": 0,
        "top5_crew_nblockbuster": 0,
        "top5_crew_nprofit": 0,
        "n_experienced_crew": 0,

        "has_director": 0,
        "director_nmovies": 0,
        "director_pop": 0,
        "director_wavg_mrating": 0,
        "director_avg_mpop": 0,
        "director_recent_mpop": 0,
        "director_nblockbuster": 0,
        "director_nprofit": 0,
        "director_career_span": 0
    }

    if not cast.empty:
        top5_cast = cast.nlargest(5, "prev_avg_pop")

        features["has_cast"] = 1
        features["top_cast_pop"] = cast["prev_avg_pop"].max()
        features["top_cast_career_span"] = cast["prev_career_span"].max()

        features["top5_cast_pop"] = top5_cast["prev_avg_pop"].nlargest(5).mean()
        features["top5_cast_wavg_mrating"] = top5_cast["prev_wavg_rating"].nlargest(5).mean()
        features["top5_cast_avg_mpop"] = top5_cast["prev_avg_mpop"].nlargest(5).mean()
        features["top5_cast_nblockbuster"] = top5_cast["prev_n_blockbuster"].nlargest(5).sum()
        features["top5_cast_nprofit"] = top5_cast["prev_n_profit"].nlargest(5).sum()
        features["top5_cast_career_span"] = top5_cast["prev_career_span"].nlargest(5).mean()
        features["n_experienced_cast"] = cast["prev_experienced"].sum()

    if not crew.empty:
        top5_crew = crew.nlargest(5, "prev_avg_pop")

        features["has_crew"] = 1
        features["top5_crew_wavg_mrating"] = top5_crew["prev_wavg_rating"].nlargest(5).mean()
        features["top5_crew_avg_mpop"] = top5_crew["prev_avg_mpop"].nlargest(5).mean()
        features["top5_crew_nblockbuster"] = top5_crew["prev_n_blockbuster"].nlargest(5).sum()
        features["top5_crew_nprofit"] = top5_crew["prev_n_profit"].nlargest(5).sum()
        features["n_experienced_crew"] = crew["prev_experienced"].sum()

    if not director.empty:
        # If there are multiple directors, choose the one with highest previous popularity
        main_director = director.nlargest(1, "prev_avg_pop").iloc[0]

        features["has_director"] = 1
        features["director_nmovies"] = main_director["prev_n_movies"]
        features["director_pop"] = main_director["prev_avg_pop"]
        features["director_wavg_mrating"] = main_director["prev_wavg_rating"]
        features["director_avg_mpop"] = main_director["prev_avg_mpop"]
        features["director_nblockbuster"] = main_director["prev_n_blockbuster"]
        features["director_nprofit"] = main_director["prev_n_profit"]
        features["director_career_span"] = main_director["prev_career_span"]
        features["director_recent_mpop"] = main_director["prev_recent_mpop"]

    return pd.Series(features)

In [21]:
movies_extras = (
    person_movie_history.groupby("Movie ID", group_keys=False)
      .apply(make_movies_cast_crew_statistics)
      .reset_index(drop=True)
)

In [22]:
# Note: We separate so we can get fresh movies_enrich and add few extra columns

movies_enrich = movies.merge(movies_extras, left_on="movie_id", right_on = "Movie ID", how="inner")

movies_enrich["Release Date"] = pd.to_datetime(movies_enrich["Release Date"], errors="coerce")
movies_enrich["Year"] = movies_enrich["Release Date"].dt.year
movies_enrich["Time_Post Digital"] = movies_enrich["Year"] >= 2010
movies_enrich["Time_Pre-Covid"] = movies_enrich["Year"] <= 2020
movies_enrich["Time_Post Netflix Expansion"] = movies_enrich["Year"] >= 2016

## Save New Dataset

In [23]:
person.to_csv("../data/processed/person.csv")
movies_enrich.to_csv("../data/processed/movies_enrich.csv")
person_movie_history.to_csv("../data/processed/person_movie_history.csv")